<a href="https://colab.research.google.com/github/AsifaBatool/Project4-AsifaBatool-Image-text-Recognition-decodelabs/blob/main/Image_text_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install pytesseract if not already installed
!pip install pytesseract

# Install Tesseract-OCR itself (if not already installed)
!sudo apt update
!sudo apt install tesseract-ocr

"""
Image Text Recognition using OCR (Optical Character Recognition)

This module implements a complete OCR pipeline that:
1. Loads an image
2. Converts to grayscale
3. Applies Gaussian blur for noise reduction
4. Applies adaptive thresholding
5. Extracts text using Tesseract OCR
6. Saves results to files

Author: AI Computer Vision Engineer
Date: 2024
"""

import cv2
import numpy as np
import pytesseract
import matplotlib.pyplot as plt
import os
from pathlib import Path


class OCRProcessor:
    """
    A professional OCR processor class that handles image preprocessing
    and text extraction using Tesseract OCR engine.
    """

    def __init__(self, image_path, output_dir="output"):
        """
        Initialize the OCR processor.

        Args:
            image_path (str): Path to the input image file
            output_dir (str): Directory to save output files (default: "output")
        """
        self.image_path = image_path
        self.output_dir = output_dir
        self.image = None
        self.gray = None
        self.blurred = None
        self.threshold = None
        self.extracted_text = None

        # Tesseract configuration
        self.config = r'--oem 3 --psm 6'

        # Create output directory if it doesn't exist
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)

    def load_image(self):
        """
        Load the image from the specified path.

        Returns:
            bool: True if image loaded successfully, False otherwise
        """
        try:
            if not os.path.exists(self.image_path):
                raise FileNotFoundError(f"Image file not found: {self.image_path}")

            self.image = cv2.imread(self.image_path)

            if self.image is None:
                raise ValueError(f"Failed to read image: {self.image_path}")

            print(f"✓ Image loaded successfully: {self.image_path}")
            print(f"  Image dimensions: {self.image.shape}")
            return True

        except FileNotFoundError as e:
            print(f"✗ Error: {e}")
            return False
        except Exception as e:
            print(f"✗ Unexpected error loading image: {e}")
            return False

    def convert_to_grayscale(self):
        """
        Convert the image to grayscale.

        Returns:
            bool: True if conversion successful, False otherwise
        """
        try:
            if self.image is None:
                raise ValueError("Image not loaded. Call load_image() first.")

            self.gray = cv2.cvtColor(self.image, cv2.COLOR_BGR2GRAY)
            print("✓ Image converted to grayscale")
            return True

        except Exception as e:
            print(f"✗ Error during grayscale conversion: {e}")
            return False

    def apply_gaussian_blur(self, kernel_size=(5, 5)):
        """
        Apply Gaussian blur to reduce noise.

        Args:
            kernel_size (tuple): Kernel size for Gaussian blur (default: (5, 5))

        Returns:
            bool: True if blur applied successfully, False otherwise
        """
        try:
            if self.gray is None:
                raise ValueError("Grayscale image not available. Call convert_to_grayscale() first.")

            self.blurred = cv2.GaussianBlur(self.gray, kernel_size, 0)
            print(f"✓ Gaussian blur applied with kernel size: {kernel_size}")
            return True

        except Exception as e:
            print(f"✗ Error during Gaussian blur: {e}")
            return False

    def apply_adaptive_threshold(self, block_size=11, constant=2):
        """
        Apply adaptive thresholding to binary image.

        Args:
            block_size (int): Block size for adaptive threshold (default: 11)
            constant (float): Constant subtracted from mean (default: 2)

        Returns:
            bool: True if thresholding successful, False otherwise
        """
        try:
            if self.blurred is None:
                raise ValueError("Blurred image not available. Call apply_gaussian_blur() first.")

            self.threshold = cv2.adaptiveThreshold(
                self.blurred,
                255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                block_size,
                constant
            )
            print(f"✓ Adaptive thresholding applied")
            print(f"  Block size: {block_size}, Constant: {constant}")
            return True

        except Exception as e:
            print(f"✗ Error during adaptive thresholding: {e}")
            return False

    def extract_text(self):
        """
        Extract text from the preprocessed image using Tesseract OCR.

        Returns:
            bool: True if text extraction successful, False otherwise
        """
        try:
            if self.threshold is None:
                raise ValueError("Threshold image not available. Call apply_adaptive_threshold() first.")

            # Check if Tesseract is installed
            try:
                pytesseract.get_tesseract_version()
            except pytesseract.TesseractNotFoundError:
                raise RuntimeError(
                    "Tesseract OCR is not installed. "
                    "Please install it: https://github.com/UB-Mannheim/tesseract/wiki"
                )

            self.extracted_text = pytesseract.image_to_string(
                self.threshold,
                config=self.config
            )

            if not self.extracted_text.strip():
                print("⚠ Warning: No text extracted from image")
            else:
                print(f"✓ Text extraction completed successfully")
                print(f"  Characters extracted: {len(self.extracted_text)}")

            return True

        except RuntimeError as e:
            print(f"✗ Tesseract Error: {e}")
            return False
        except Exception as e:
            print(f"✗ Error during text extraction: {e}")
            return False

    def save_preprocessing_results(self):
        """
        Save preprocessed images to the output directory.

        Returns:
            bool: True if all images saved successfully, False otherwise
        """
        try:
            success = True

            # Save grayscale image
            if self.gray is not None:
                gray_path = os.path.join(self.output_dir, "gray.jpg")
                cv2.imwrite(gray_path, self.gray)
                print(f"✓ Grayscale image saved: {gray_path}")

            # Save threshold image
            if self.threshold is not None:
                threshold_path = os.path.join(self.output_dir, "threshold.jpg")
                cv2.imwrite(threshold_path, self.threshold)
                print(f"✓ Threshold image saved: {threshold_path}")

            return success

        except Exception as e:
            print(f"✗ Error saving preprocessing results: {e}")
            return False

    def save_extracted_text(self):
        """
        Save extracted text to a file.

        Returns:
            bool: True if text saved successfully, False otherwise
        """
        try:
            if self.extracted_text is None:
                raise ValueError("No text to save. Call extract_text() first.")

            result_path = os.path.join(self.output_dir, "result.txt")

            with open(result_path, 'w', encoding='utf-8') as f:
                f.write(self.extracted_text)

            print(f"✓ Extracted text saved: {result_path}")
            return True

        except Exception as e:
            print(f"✗ Error saving extracted text: {e}")
            return False

    def display_results(self):
        """
        Display the results using matplotlib visualization.

        Returns:
            bool: True if display successful, False otherwise
        """
        try:
            if self.image is None or self.gray is None or self.threshold is None:
                raise ValueError("Not all preprocessing steps completed.")

            # Convert BGR to RGB for matplotlib display
            image_rgb = cv2.cvtColor(self.image, cv2.COLOR_BGR2RGB)

            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            fig.suptitle('OCR Preprocessing Pipeline', fontsize=16, fontweight='bold')

            # Original image
            axes[0, 0].imshow(image_rgb)
            axes[0, 0].set_title('Original Image')
            axes[0, 0].axis('off')

            # Grayscale image
            axes[0, 1].imshow(self.gray, cmap='gray')
            axes[0, 1].set_title('Grayscale Image')
            axes[0, 1].axis('off')

            # Blurred image
            axes[1, 0].imshow(self.blurred, cmap='gray')
            axes[1, 0].set_title('Gaussian Blur (5x5)')
            axes[1, 0].axis('off')

            # Threshold image
            axes[1, 1].imshow(self.threshold, cmap='gray')
            axes[1, 1].set_title('Adaptive Threshold (Gaussian C)')
            axes[1, 1].axis('off')

            plt.tight_layout()
            plt.show()

            return True

        except Exception as e:
            print(f"✗ Error displaying results: {e}")
            return False

    def print_extracted_text(self):
        """
        Print the extracted text in a formatted way.
        """
        print("\n" + "="*50)
        print("EXTRACTED TEXT")
        print("="*50)

        if self.extracted_text:
            print(self.extracted_text)
        else:
            print("No text extracted from image.")

        print("="*50 + "\n")

    def process(self):
        """
        Execute the complete OCR processing pipeline.

        Returns:
            bool: True if entire pipeline succeeds, False otherwise
        """
        print("\n" + "="*60)
        print("STARTING OCR PROCESSING PIPELINE")
        print("="*60 + "\n")

        # Step 1: Load image
        if not self.load_image():
            return False

        # Step 2: Convert to grayscale
        if not self.convert_to_grayscale():
            return False

        # Step 3: Apply Gaussian blur
        if not self.apply_gaussian_blur(kernel_size=(5, 5)):
            return False

        # Step 4: Apply adaptive thresholding
        if not self.apply_adaptive_threshold(block_size=11, constant=2):
            return False

        # Step 5: Extract text using OCR
        if not self.extract_text():
            return False

        # Step 6: Save preprocessing results
        if not self.save_preprocessing_results():
            return False

        # Step 7: Save extracted text
        if not self.save_extracted_text():
            return False

        # Step 8: Print extracted text
        self.print_extracted_text()

        # Step 9: Display results
        print("Display visualization? (Optional)")
        # Uncomment the line below to display visualization
        # self.display_results()

        print("="*60)
        print("OCR PROCESSING PIPELINE COMPLETED SUCCESSFULLY")
        print("="*60 + "\n")

        return True


def main():
    """
    Main function to demonstrate the OCR processor.
    """

    # Create images directory if it doesn't exist
    if not os.path.exists("images"):
        os.makedirs("images")

    image_urls = [
        "https://i.stack.imgur.com/pbK1C.png", # A working example
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/sudoku.png" # Another working example
    ]
    image_file_path = "images/sample.png" # Changed to .png as new urls are png
    image_downloaded_successfully = False

    for url in image_urls:
        # Check if the file exists AND is readable by OpenCV, otherwise try downloading
        if not os.path.exists(image_file_path) or cv2.imread(image_file_path) is None:
            print(f"Attempting to download sample image from {url}...")
            try:
                import requests
                response = requests.get(url)
                response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
                with open(image_file_path, "wb") as f:
                    f.write(response.content)
                print("Sample image downloaded.")

                # Verify if the downloaded image can be read by OpenCV
                test_image = cv2.imread(image_file_path)
                if test_image is None:
                    print(f"✗ Warning: Downloaded image from {url} could not be read by OpenCV. Trying next URL if available.")
                    if os.path.exists(image_file_path):
                        os.remove(image_file_path) # Remove potentially corrupted file
                else:
                    print(f"✓ Image from {url} successfully read by OpenCV.")
                    image_downloaded_successfully = True
                    break # Exit loop if image is successfully downloaded and readable
            except requests.exceptions.RequestException as e:
                print(f"✗ Error downloading image from {url}: {e}. Trying next URL if available.")
            except Exception as e:
                print(f"✗ An unexpected error occurred during download/verification from {url}: {e}. Trying next URL if available.")
        else:
            # If the image already exists and is readable, we don't need to download/re-download
            test_image = cv2.imread(image_file_path)
            if test_image is not None:
                print(f"✓ Using existing and readable sample image: {image_file_path}")
                image_downloaded_successfully = True
                break
            else:
                print(f"✗ Existing image {image_file_path} is corrupted or unreadable. Attempting to re-download.")
                if os.path.exists(image_file_path):
                    os.remove(image_file_path) # Remove corrupted file and try to re-download

    if not image_downloaded_successfully:
        print("✗ Failed to download or read any sample image. Please provide a valid image in 'images/sample.png'.")
        return # Exit main if no image is available

    # Create OCR processor instance
    processor = OCRProcessor(image_file_path, output_dir="output")

    # Execute the complete pipeline
    success = processor.process()

    if success:
        print("✓ All operations completed successfully!")
    else:
        print("✗ Pipeline execution failed. Please check error messages above.")


if __name__ == "__main__":
    main()

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
19 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree